# Code mẫu — Speech recognition

Âm thanh và bản chép lời dài ngắn khác nhau, nên không thể gán đại mỗi mẫu âm thanh một chữ. Các ví dụ dưới đây tách rõ ba việc: gộp một đường đi CTC thành chữ, cộng mọi đường đi để tính loss, và chấm bản chép lời bằng WER. Phần cuối kiểm sample rate trước khi đưa âm thanh vào bộ tiền xử lý.

## Ví dụ 1: Ký hiệu trống giữ được hai chữ giống nhau liền nhau

CTC cho model đoán một ký hiệu ở mỗi frame, kể cả một ký hiệu trống, gọi là blank. Ta dùng hai chữ `a`, `b`, và ký hiệu trống `_`. Chuỗi ký hiệu theo từng frame gọi là một đường đi. Các đường đi dưới đây được đặt bằng tay, chưa phải dự đoán của model nào.

In [ ]:
def ctc_collapse(path, blank="_"):
    transcript = []
    previous = None
    for token in path:
        if token != previous and token != blank:
            transcript.append(token)
        previous = token  # Nhớ cả ký hiệu trống, để phân biệt a_a với aa.
    return "".join(transcript)

paths = ["aaa", "a_a", "aa_bb_b", "___", ""]
for path in paths:
    print(f"{path!r} -> {ctc_collapse(path)!r}")

assert ctc_collapse("aa") == "a"
assert ctc_collapse("a_a") == "aa"
assert ctc_collapse("aa_bb_b") == "abb"

'aaa' -> 'a'
'a_a' -> 'aa'
'aa_bb_b' -> 'abb'
'___' -> ''
'' -> ''


Quy tắc có hai bước, theo đúng thứ tự: gộp các ký hiệu giống nhau đứng liền nhau, rồi bỏ ký hiệu trống. `aaa` cho một chữ `a`, vì ba chữ liền nhau được gộp lại. `a_a` cho hai chữ `a`, vì ký hiệu trống chen giữa nên chúng không được gộp.

Nếu bỏ ký hiệu trống trước rồi mới gộp, `a_a` sẽ thành `aa` rồi thành `a`, giống hệt `aaa`. Mọi từ có hai chữ cái giống nhau liền nhau đều không viết ra được. Đường đi toàn ký hiệu trống, hay đường đi rỗng, đều cho chuỗi rỗng.

## Ví dụ 2: Tám đường đi cùng góp vào CTC loss

Có ba frame. Ở mỗi frame, model cho hai xác suất, theo thứ tự ký hiệu trống và chữ `a`: `[0.6, 0.4]`, `[0.5, 0.5]`, `[0.4, 0.6]`. Ta liệt kê đủ cả tám đường đi, cộng xác suất theo bản chép lời mà mỗi đường đi gộp ra, rồi so với `CTCLoss` của PyTorch.

In [ ]:
import numpy as np
import torch
from itertools import product

torch.set_num_threads(1)
probabilities = np.array([[0.6, 0.4], [0.5, 0.5], [0.4, 0.6]])

def collapse_ids(path):
    return tuple(token for i, token in enumerate(path)
                 if token != 0 and (i == 0 or token != path[i - 1]))

totals = {}
for path in product([0, 1], repeat=3):
    transcript = collapse_ids(path)
    path_probability = np.prod(probabilities[np.arange(3), path])
    totals[transcript] = totals.get(transcript, 0.0) + path_probability

# Đường ___ và đường a_a, mỗi đường có xác suất 0.12. Sáu đường còn lại đều gộp thành a.
assert np.allclose([totals[()], totals[(1,)], totals[(1, 1)]], [0.12, 0.76, 0.12])
assert np.isclose(sum(totals.values()), 1)
log_probs = torch.tensor(probabilities, dtype=torch.float64).log()[:, None, :]
criterion = torch.nn.CTCLoss(blank=0, reduction="sum")
loss = criterion(log_probs, torch.tensor([1]), torch.tensor([3]), torch.tensor([1]))
assert np.isclose(loss.item(), -np.log(0.76))
too_short = criterion(log_probs[:2], torch.tensor([1, 1]), torch.tensor([2]), torch.tensor([2]))

print(f"P(rong)={totals[()]:.2f}, P(a)={totals[(1,)]:.2f}, P(aa)={totals[(1, 1)]:.2f}")
print("log_probs (frame, batch, class):", tuple(log_probs.shape))
print(f"CTCLoss = {loss.item():.4f}")
print("target aa chi co 2 frame, loss vo han:", torch.isinf(too_short).item())

P(rong)=0.12, P(a)=0.76, P(aa)=0.12
log_probs (frame, batch, class): (3, 1, 2)
CTCLoss = 0.2744
target aa chi co 2 frame, loss vo han: True


Tám đường đi chia làm ba nhóm. Đường `___` cho chuỗi rỗng, với xác suất $0.6\cdot0.5\cdot0.4=0.12$. Đường `a_a` cho `aa`, cũng 0.12. Sáu đường còn lại đều gộp thành `a`, tổng xác suất 0.76. CTC loss cho nhãn `a` chính là $-\log 0.76\approx0.2744$, khớp với PyTorch. Model không cần biết chữ `a` nằm ở frame nào, vì loss cộng mọi cách sắp xếp hợp lệ.

`CTCLoss` nhận log xác suất có shape `(T, B, C)`: frame, batch, số lớp. Nhãn đích không được chứa ký hiệu trống. Dòng cuối cho thấy một giới hạn. Nhãn `aa` cần ít nhất ba frame, `a _ a`, nên với hai frame thì không có đường đi hợp lệ nào, và loss là vô cùng. Đặt loss vô cùng thành 0 chỉ che lỗi đi, chứ không tạo ra đường đi hợp lệ.

## Ví dụ 3: WER trên cả tập, và một lần sửa sai nội dung

WER là tỉ lệ lỗi theo từ: số lần sửa ít nhất, gồm thay, xóa, chèn, để biến bản đoán thành bản chuẩn, chia cho số từ của bản chuẩn. Bộ chấm ở đây đưa về chữ thường rồi tách theo khoảng trắng, giữ nguyên dấu tiếng Việt. Câu ngắn bị thiếu chữ `AI`, câu dài thì đúng hoàn toàn. Ta xem vì sao lấy trung bình WER từng câu lại khác WER tính trên cả tập. Ví dụ chỉ chấm các chuỗi có sẵn, không gọi model nhận dạng nào.

In [ ]:
import numpy as np

def tokens(text):
    return text.lower().split()

def edit_distance(reference, prediction):
    previous = list(range(len(prediction) + 1))
    for row, reference_token in enumerate(reference, start=1):
        current = [row]
        for column, predicted_token in enumerate(prediction, start=1):
            current.append(min(
                previous[column] + 1,      # Xóa một từ của bản chuẩn.
                current[column - 1] + 1,   # Chèn một từ của bản đoán.
                previous[column - 1] + (reference_token != predicted_token),
            ))
        previous = current
    return previous[-1]

references = ["em học AI", "hôm nay lớp học cách chấm kết quả ai"]
predictions = ["em học", "hôm nay lớp học cách chấm kết quả ai"]
counts = [edit_distance(tokens(r), tokens(p)) for r, p in zip(references, predictions)]
lengths = [len(tokens(r)) for r in references]
assert counts == [1, 0] and lengths == [3, 9]
assert edit_distance([], ["ai", "ai"]) == 2
assert edit_distance(["ai"], []) == 1
assert sum(lengths) > 0  # Tập rỗng cần một quy ước riêng cho metric.

print("loi/token reference:", list(zip(counts, lengths)))
print(f"corpus WER = {sum(counts) / sum(lengths):.3f}")
print(f"mean WER cau = {np.mean(np.array(counts) / lengths):.3f}")
reference = tokens("mai thi ai")
for name, prediction in [("goc", "mai thi ai"), ("sua", "mai thi toán")]:
    print(f"{name}: WER = {edit_distance(reference, tokens(prediction)) / len(reference):.3f}")
many_insertions = edit_distance(tokens("ai"), tokens("ai học thêm hôm nay"))
print(f"nhieu token chen: WER = {many_insertions / 1:.3f}")

loi/token reference: [(1, 3), (0, 9)]
corpus WER = 0.083
mean WER cau = 0.167
goc: WER = 0.000
sua: WER = 0.333
nhieu token chen: WER = 4.000


Cả tập có 1 lỗi trên 12 từ, nên WER là khoảng 0.083. Lấy trung bình hai câu thì ra $(1/3+0)/2\approx0.167$, gấp đôi. Cách trung bình theo câu làm câu ngắn nặng ngang câu dài, nên một lỗi ở câu ngắn bị phóng to. Phải dùng đúng cách tính mà đề quy định.

Đổi `ai` thành `toán` làm câu đúng thành câu có WER 0.333. Câu mới vẫn trôi chảy, nghe rất tự nhiên, nhưng sai nội dung. Một model ngôn ngữ “sửa” câu cho hay hơn có thể gây ra đúng lỗi này. Dòng cuối có bốn từ bị chèn thêm, trên bản chuẩn chỉ có một từ, nên WER bằng 4. WER có thể lớn hơn 1, nên nó không phải là phần trăm câu sai.

## Ví dụ 4: Phát hiện sample rate sai trước bước tiền xử lý

Một âm 1000 Hz được sinh với 16 000 mẫu mỗi giây, gồm 1600 mẫu. Nếu khai nhầm sample rate là 8000, cùng mảng ấy sẽ bị hiểu thành một clip dài gấp đôi, với cao độ chỉ còn 500 Hz. Ở đây ta biết chính xác dữ liệu được sinh ra sao, nên sửa được thông tin khai báo. Với file thu âm thật không rõ nguồn, phải đọc và kiểm phần đầu file. Nếu cần đổi sample rate, phải resample có lọc, không chỉ đổi con số khai báo.

In [ ]:
import numpy as np

actual_rate = 16000
waveform = np.sin(2 * np.pi * 1000 * np.arange(1600) / actual_rate)
peak_bin = np.abs(np.fft.rfft(waveform)).argmax()

def check_audio(signal, declared_rate, expected_rate):
    if signal.ndim != 1 or not np.isfinite(signal).all():
        raise ValueError("waveform can mono va huu han")
    if declared_rate != expected_rate:
        raise ValueError("sample rate chua khop hop dong processor")
    return signal

for declared_rate in [8000, 16000]:
    grid = np.fft.rfftfreq(len(waveform), d=1 / declared_rate)
    print(f"gan nhan {declared_rate} Hz: {len(waveform) / declared_rate:.3f} s, "
          f"dinh {grid[peak_bin]:.0f} Hz")
try:
    check_audio(waveform, 8000, 16000)
except ValueError as error:
    print("preflight:", error)
checked = check_audio(waveform, actual_rate, 16000)
assert checked is waveform
print("metadata dung: preflight dat")

gan nhan 8000 Hz: 0.200 s, dinh 500 Hz
gan nhan 16000 Hz: 0.100 s, dinh 1000 Hz
preflight: sample rate chua khop hop dong processor
metadata dung: preflight dat


Số mẫu vẫn là 1600, nhưng khai nhầm 8000 thì clip bị hiểu là dài 0.2 giây thay vì 0.1 giây, và đỉnh phổ rơi vào 500 Hz thay vì 1000 Hz. Một model nhận dạng giọng nói được train ở 16 000 Hz sẽ nghe thấy một giọng trầm và chậm gấp đôi.

Hàm kiểm trước khi chạy bắt buộc sample rate khai báo phải khớp với yêu cầu của bộ tiền xử lý. Nhưng nó không phát hiện được trường hợp phần đầu file khai sai mà con số lại trùng với yêu cầu. Các phép kiểm này chỉ bảo vệ đầu vào. Chúng chưa tạo ra bản chép lời nào, và chưa đo chất lượng của một model nhận dạng giọng nói pretrained.